# 01 — Phase 1 Local Baselines

Curated starter notebook for local baseline analysis.

This notebook uses local expected-result fixtures under `results/expected/` to quickly compare defense behavior under clean vs backdoor settings and visualize trajectory differences.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

EXPECTED_DIR = ROOT / "results" / "expected"
assert EXPECTED_DIR.exists(), f"Missing expected fixtures at {EXPECTED_DIR}"


def load_json(path: Path) -> dict:
    return json.loads(path.read_text())

In [ ]:
# Load clean + backdoor fixtures across defenses.
clean_files = sorted(EXPECTED_DIR.glob("akash_clean_*_3c.json"))
backdoor_files = sorted(EXPECTED_DIR.glob("akash_backdoor_*_3c.json"))

assert clean_files, "No clean fixtures found"
assert backdoor_files, "No backdoor fixtures found"

summary = []
for path in backdoor_files:
    payload = load_json(path)
    defense = payload["config"]["defense"]
    final = payload["rounds"][-1]
    summary.append({
        "defense": defense,
        "final_ppl": float(final["perplexity"]),
        "final_asr": float(final.get("asr", 0.0)),
        "final_agg_time": float(final["agg_time"]),
    })

summary = sorted(summary, key=lambda x: x["defense"])
summary

In [ ]:
# Plot final perplexity and ASR by defense for backdoor fixtures.
defenses = [row["defense"] for row in summary]
ppl = [row["final_ppl"] for row in summary]
asr = [row["final_asr"] for row in summary]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(defenses, ppl, color="#4C78A8")
axes[0].set_title("Backdoor: Final Perplexity by Defense")
axes[0].set_ylabel("Perplexity")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(defenses, asr, color="#E45756")
axes[1].set_title("Backdoor: Final ASR by Defense")
axes[1].set_ylabel("ASR")
axes[1].tick_params(axis="x", rotation=30)
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("Phase 1 Local Baseline Snapshot (Expected Fixtures)", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Compare clean vs backdoor trajectory for FedAvg fixture.
clean_fedavg = load_json(EXPECTED_DIR / "akash_clean_FedAvg_3c.json")
backdoor_fedavg = load_json(EXPECTED_DIR / "akash_backdoor_FedAvg_3c.json")

clean_rounds = clean_fedavg["rounds"]
backdoor_rounds = backdoor_fedavg["rounds"]

x_clean = [r["round"] for r in clean_rounds]
ppl_clean = [r["perplexity"] for r in clean_rounds]

x_backdoor = [r["round"] for r in backdoor_rounds]
ppl_backdoor = [r["perplexity"] for r in backdoor_rounds]
asr_backdoor = [r.get("asr", 0.0) for r in backdoor_rounds]

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(x_clean, ppl_clean, marker="o", color="#4C78A8", label="Clean PPL")
ax1.plot(x_backdoor, ppl_backdoor, marker="s", color="#F58518", label="Backdoor PPL")
ax1.set_xlabel("Round")
ax1.set_ylabel("Perplexity")
ax1.grid(alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(x_backdoor, asr_backdoor, marker="^", color="#E45756", label="Backdoor ASR")
ax2.set_ylabel("ASR")

lines = ax1.get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc="upper right")

plt.title("FedAvg Fixture: Utility vs Attack Trajectory")
plt.tight_layout()
plt.show()

## Conclusion

- This notebook provides a quick baseline lens for local defense/attack behavior.
- Next step for paper-quality charts: swap expected fixtures with exported W&B histories for canonical local runs.